# 01 — Data Exploration

**Purpose:** Explore and profile the five Gold Lakehouse tables produced by the
BDC `personalize-service` analytics pipeline. These tables form the primary
dataset for all downstream recommender models.

## Gold Tables
| Table | Description |
|-------|-------------|
| `gold_student_course_metrics` | Per-student, per-course accuracy and completion metrics |
| `gold_concept_struggles` | Per-student, per-node struggle rate and attempt counts |
| `gold_user_item_matrix` | Implicit affinity scores for collaborative filtering |
| `gold_struggle_alerts` | Real-time struggle threshold alerts |
| `gold_study_recommendations` | Heuristic study action recommendations |

**Data access path:** `./data/lakehouse/gold/<table>.parquet`  
**Fallback:** REST API via `http://localhost:8085`

In [ ]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import requests

warnings.filterwarnings('ignore')
%matplotlib inline

# Add parent directory to path so scripts/ is importable
sys.path.insert(0, os.path.abspath('..'))

GOLD_DIR = os.environ.get('BDC_GOLD_DIR', '../data/lakehouse/gold')
API_BASE = os.environ.get('BDC_API_BASE', 'http://localhost:8085')
AI_SECRET = os.environ.get('AI_SECRET', '')

GOLD_TABLES = [
    'gold_student_course_metrics',
    'gold_concept_struggles',
    'gold_user_item_matrix',
    'gold_struggle_alerts',
    'gold_study_recommendations',
]

print('Gold directory:', GOLD_DIR)
print('API base:', API_BASE)

## 1. Load Gold Tables

Try Parquet first. If the file is missing (e.g., export has not been run),
fall back to the REST API endpoint.

In [ ]:
def load_gold_table(table_name: str) -> pd.DataFrame:
    """Load a Gold table from Parquet, with REST API fallback."""
    path = os.path.join(GOLD_DIR, f'{table_name}.parquet')
    if os.path.exists(path):
        print(f'[Parquet] Loading {table_name}...')
        return pd.read_parquet(path)

    print(f'[API Fallback] {table_name} not found at {path}. Trying REST API...')
    endpoint_map = {
        'gold_user_item_matrix':       '/personalize/analytics/gold/interaction-matrix',
        'gold_concept_struggles':      '/personalize/analytics/gold/concept-struggles',
        'gold_student_course_metrics': '/personalize/analytics/gold/student-metrics',
        'gold_struggle_alerts':        '/personalize/analytics/gold/struggle-alerts',
        'gold_study_recommendations':  '/personalize/analytics/gold/study-recommendations',
    }
    endpoint = endpoint_map.get(table_name)
    if endpoint is None:
        raise ValueError(f'Unknown table: {table_name}')
    headers = {'X-AI-Secret': AI_SECRET} if AI_SECRET else {}
    try:
        resp = requests.get(f'{API_BASE}{endpoint}', headers=headers, timeout=30)
        resp.raise_for_status()
        return pd.DataFrame(resp.json())
    except Exception as e:
        print(f'  API call failed: {e}')
        print('  Generating synthetic demo data for exploration...')
        return _generate_demo_data(table_name)


def _generate_demo_data(table_name: str) -> pd.DataFrame:
    """Generate synthetic demo data when no data source is available."""
    rng = np.random.default_rng(42)
    n_users, n_nodes, n_courses = 200, 80, 10
    user_ids = list(range(1, n_users + 1))
    node_ids = list(range(100, 100 + n_nodes))
    course_ids = list(range(1, n_courses + 1))
    action_types = ['view', 'quick_check', 'learn', 'review', 'hint_request']

    if table_name == 'gold_user_item_matrix':
        n = 2000
        return pd.DataFrame({
            'user_id': rng.choice(user_ids, n),
            'node_id': rng.choice(node_ids, n),
            'action_type': rng.choice(action_types, n),
            'implicit_affinity_score': rng.uniform(0.1, 5.0, n).round(3),
            'interaction_count': rng.integers(1, 20, n),
        })
    elif table_name == 'gold_concept_struggles':
        n = 1500
        return pd.DataFrame({
            'user_id': rng.choice(user_ids, n),
            'node_id': rng.choice(node_ids, n),
            'course_id': rng.choice(course_ids, n),
            'struggle_rate': rng.uniform(0.0, 1.0, n).round(3),
            'attempt_count': rng.integers(1, 30, n),
            'incorrect_count': rng.integers(0, 15, n),
        })
    elif table_name == 'gold_student_course_metrics':
        n = 500
        return pd.DataFrame({
            'user_id': rng.choice(user_ids, n),
            'course_id': rng.choice(course_ids, n),
            'check_accuracy': rng.uniform(0.0, 1.0, n).round(3),
            'completion_rate': rng.uniform(0.0, 1.0, n).round(3),
            'total_attempts': rng.integers(1, 50, n),
        })
    elif table_name == 'gold_struggle_alerts':
        n = 300
        return pd.DataFrame({
            'user_id': rng.choice(user_ids, n),
            'node_id': rng.choice(node_ids, n),
            'alert_level': rng.choice(['low', 'medium', 'high'], n),
            'struggle_rate': rng.uniform(0.5, 1.0, n).round(3),
        })
    else:  # gold_study_recommendations
        actions = ['review_struggle_concept', 'discuss_with_ai', 'learn_next_lesson']
        n = 400
        return pd.DataFrame({
            'user_id': rng.choice(user_ids, n),
            'course_id': rng.choice(course_ids, n),
            'recommended_action_type': rng.choice(actions, n),
            'recommended_node_id': rng.choice(node_ids, n),
        })


# Load all tables
tables = {}
for t in GOLD_TABLES:
    tables[t] = load_gold_table(t)
    print(f'  -> shape: {tables[t].shape}')

print('\nAll tables loaded.')

## 2. Profile Each DataFrame

For each Gold table: shape, dtypes, descriptive statistics, and null counts.

In [ ]:
for name, df in tables.items():
    print('=' * 70)
    print(f'TABLE: {name}')
    print('=' * 70)
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
    print('\nDtypes:')
    print(df.dtypes.to_string())
    print('\nDescriptive statistics:')
    display(df.describe(include='all').T)
    null_counts = df.isnull().sum()
    print('\nNull counts:')
    print(null_counts[null_counts > 0].to_string() if null_counts.any() else '  No nulls found.')
    print()

## 3. Interaction Count by Action Type

Bar chart from `gold_user_item_matrix` — shows which student actions are most frequent.

In [ ]:
df_matrix = tables['gold_user_item_matrix']

if 'action_type' in df_matrix.columns:
    action_counts = df_matrix.groupby('action_type')['interaction_count'].sum().sort_values(ascending=False) \
        if 'interaction_count' in df_matrix.columns \
        else df_matrix['action_type'].value_counts()

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(action_counts)))
    bars = ax.bar(action_counts.index, action_counts.values, color=colors, edgecolor='white', linewidth=0.8)

    for bar, val in zip(bars, action_counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + action_counts.max() * 0.01,
                f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title('Interaction Count by Action Type\n(gold_user_item_matrix)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Action Type', fontsize=12)
    ax.set_ylabel('Total Interaction Count', fontsize=12)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.savefig('../output/exploration_action_type_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to ../output/exploration_action_type_bar.png')
else:
    print('Column action_type not found in gold_user_item_matrix')

## 4. Top-20 Nodes by Struggle Rate

Bar chart from `gold_concept_struggles` — highlights which knowledge nodes
students find most difficult.

In [ ]:
df_struggles = tables['gold_concept_struggles']

if 'node_id' in df_struggles.columns and 'struggle_rate' in df_struggles.columns:
    top_struggles = (
        df_struggles.groupby('node_id')['struggle_rate']
        .mean()
        .sort_values(ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    cmap = plt.cm.RdYlGn_r
    colors = [cmap(v) for v in np.linspace(0.1, 0.9, len(top_struggles))]
    bars = ax.barh(top_struggles.index.astype(str), top_struggles.values, color=colors, edgecolor='white')

    for bar, val in zip(bars, top_struggles.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9)

    ax.set_title('Top-20 Knowledge Nodes by Average Struggle Rate\n(gold_concept_struggles)',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Average Struggle Rate', fontsize=12)
    ax.set_ylabel('Node ID', fontsize=12)
    ax.set_xlim(0, min(top_struggles.max() * 1.15, 1.0))
    ax.axvline(0.6, color='red', linestyle='--', alpha=0.6, label='60% threshold')
    ax.legend(fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig('../output/exploration_struggle_rate_bar.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to ../output/exploration_struggle_rate_bar.png')
else:
    print('Expected columns not found in gold_concept_struggles')

## 5. Summary Statistics

In [ ]:
print('=== BDC Lakehouse Summary Statistics ===')

# Unique users
users_sources = []
for t in ['gold_user_item_matrix', 'gold_student_course_metrics', 'gold_concept_struggles']:
    df = tables.get(t)
    if df is not None and 'user_id' in df.columns:
        users_sources.append(set(df['user_id'].unique()))
unique_users = len(set.union(*users_sources)) if users_sources else 0
print(f'Unique users (across all tables):  {unique_users:,}')

# Unique courses
courses_sources = []
for t in ['gold_student_course_metrics', 'gold_concept_struggles', 'gold_study_recommendations']:
    df = tables.get(t)
    if df is not None and 'course_id' in df.columns:
        courses_sources.append(set(df['course_id'].unique()))
unique_courses = len(set.union(*courses_sources)) if courses_sources else 0
print(f'Unique courses (across all tables): {unique_courses:,}')

# Unique nodes
nodes_sources = []
for t in ['gold_user_item_matrix', 'gold_concept_struggles', 'gold_struggle_alerts']:
    df = tables.get(t)
    if df is not None and 'node_id' in df.columns:
        nodes_sources.append(set(df['node_id'].unique()))
unique_nodes = len(set.union(*nodes_sources)) if nodes_sources else 0
print(f'Unique knowledge nodes:             {unique_nodes:,}')

# Total interactions
df_m = tables.get('gold_user_item_matrix')
if df_m is not None:
    total_interactions = df_m['interaction_count'].sum() if 'interaction_count' in df_m.columns else len(df_m)
    print(f'Total interaction events:           {int(total_interactions):,}')
    if 'implicit_affinity_score' in df_m.columns:
        print(f'Mean implicit affinity score:       {df_m["implicit_affinity_score"].mean():.4f}')

print()
print('Gold table sizes:')
for name, df in tables.items():
    print(f'  {name:<40}: {df.shape[0]:>6,} rows x {df.shape[1]} cols')